In [8]:
from dotenv import load_dotenv

# Load OPENAI_API_KEY (and any other secrets) from a local .env file.
load_dotenv()


True

# Notebook 2 · Hierarchical Multi-Agent System

> **Series:** LangChain MAS Architectures · Travel Agency Use Case

---

## Architecture Overview

In a **hierarchical MAS** one agent (the *manager*) controls the workflow.
It plans, delegates subtasks to *workers*, and synthesises their outputs.
Workers never communicate with each other — all routing flows through the manager.

```
              ┌──────────────┐
              │   MANAGER    │  ← plans, delegates, synthesises
              └──┬───┬───┬───┘
     ┌───────────┘   │   └───────────┐
┌────▼────┐     ┌────▼────┐     ┌────▼────┐
│  Dest   │     │Booking  │     │Activity │
│ Worker  │     │ Worker  │     │ Worker  │
└─────────┘     └─────────┘     └─────────┘
     worker → manager  (never worker → worker)
```

## Pattern in this notebook

| LangChain concept | Role in hierarchical MAS |
|---|---|
| `AgentState` | Holds `manager_notes: list[str]` — the manager's running log |
| `create_agent` (workers) | Subordinate agents that answer only the manager |
| `@tool` + `ToolRuntime` | Workers read context from state, report back |
| `Command(update=...)` | Appends each worker result to `manager_notes` |
| `create_agent` (manager) | Top-level coordinator that plans and synthesises |


## 1 · Setup

In [9]:
# ── Traveler request ─────────────────────────────────────────────────────────
USER_REQUEST = """\
Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan""".strip()

# ── Static catalog ────────────────────────────────────────────────────────────
# Agents must only use options listed here.
DESTINATIONS = {
    "Lisbon":    {"best_period": "April–June", "style": "sunny, walkable, relaxed",
                  "notes": "great for food, viewpoints, and compact neighborhoods"},
    "Barcelona": {"best_period": "April–June", "style": "lively, artistic, seaside",
                  "notes": "strong mix of architecture, beach walks, and tapas"},
    "Prague":    {"best_period": "April–June", "style": "historic, compact, lower-cost",
                  "notes": "easy sightseeing with a classic old-town atmosphere"},
}

FLIGHTS = [
    {"destination": "Lisbon",    "code": "TP-833", "price": 180, "type": "direct",  "duration": "3h 05m"},
    {"destination": "Lisbon",    "code": "IB-310", "price": 150, "type": "1 stop",  "duration": "5h 10m"},
    {"destination": "Barcelona", "code": "VY-611", "price": 140, "type": "direct",  "duration": "1h 50m"},
    {"destination": "Barcelona", "code": "IB-220", "price": 125, "type": "1 stop",  "duration": "4h 00m"},
    {"destination": "Prague",    "code": "FR-721", "price": 110, "type": "direct",  "duration": "1h 55m"},
    {"destination": "Prague",    "code": "OS-410", "price": 145, "type": "1 stop",  "duration": "3h 45m"},
]

HOTELS = [
    {"destination": "Lisbon",    "name": "Baixa Stay",       "price_per_night": 145, "style": "central boutique hotel"},
    {"destination": "Lisbon",    "name": "River Rooms",       "price_per_night": 120, "style": "simple hotel near transit"},
    {"destination": "Barcelona", "name": "Born Hotel",        "price_per_night": 160, "style": "central design hotel"},
    {"destination": "Barcelona", "name": "Gracia Inn",        "price_per_night": 130, "style": "quiet hotel in a walkable district"},
    {"destination": "Prague",    "name": "Old Town House",    "price_per_night": 115, "style": "historic hotel near main sights"},
    {"destination": "Prague",    "name": "City Garden Hotel", "price_per_night":  95, "style": "budget-friendly hotel with tram access"},
]

ACTIVITIES = [
    {"destination": "Lisbon",    "name": "Alfama food walk",                    "tag": "food",    "price": 35},
    {"destination": "Lisbon",    "name": "Belém and river tram day",            "tag": "culture", "price": 25},
    {"destination": "Barcelona", "name": "Gothic Quarter tapas evening",        "tag": "food",    "price": 40},
    {"destination": "Barcelona", "name": "Sagrada Família and modernism route", "tag": "culture", "price": 32},
    {"destination": "Prague",    "name": "Old Town walking tour",               "tag": "culture", "price": 18},
    {"destination": "Prague",    "name": "Czech dinner and jazz night",         "tag": "food",    "price": 30},
]

# ── Catalog helpers ───────────────────────────────────────────────────────────
def flights_for(destination: str)    -> list: return [f for f in FLIGHTS    if f["destination"] == destination]
def hotels_for(destination: str)     -> list: return [h for h in HOTELS     if h["destination"] == destination]
def activities_for(destination: str) -> list: return [a for a in ACTIVITIES if a["destination"] == destination]

def catalog_text() -> str:
    lines = []
    for dest, info in DESTINATIONS.items():
        lines.append(f"Destination: {dest}")
        lines.append(f"  Best period : {info['best_period']}")
        lines.append(f"  Style       : {info['style']}")
        lines.append(f"  Notes       : {info['notes']}")
        lines.append("  Flights:")
        for f in flights_for(dest):
            lines.append(f"    - {f['code']} | {f['type']} | EUR {f['price']} | {f['duration']}")
        lines.append("  Hotels:")
        for h in hotels_for(dest):
            lines.append(f"    - {h['name']} | EUR {h['price_per_night']}/night | {h['style']}")
        lines.append("  Activities:")
        for a in activities_for(dest):
            lines.append(f"    - {a['name']} | {a['tag']} | EUR {a['price']}")
        lines.append("")
    return "\n".join(lines).strip()

CATALOG_TEXT = catalog_text()

print("USER_REQUEST:")
print(USER_REQUEST)
print("\nCatalog loaded — destinations:", list(DESTINATIONS.keys()))


USER_REQUEST:
Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan

Catalog loaded — destinations: ['Lisbon', 'Barcelona', 'Prague']


## 2 · Custom State

`HierarchicalState` extends `AgentState` with one extra field:

- `manager_notes` — the manager's growing log of worker outputs.

Workers see the existing notes so later workers have context from earlier ones,
but they receive this context *via the manager*, never directly from each other.


In [10]:
from langchain.agents import AgentState
import operator
from typing import Annotated

class HierarchicalState(AgentState):
    # The manager's running log.
    # After each worker call, the manager appends the result here so
    # subsequent workers (and the final synthesis) can build on earlier work.
    manager_notes: Annotated[list[str], operator.add]


## 3 · Worker Sub-Agents

Three workers are created with `create_agent`, each with a fixed responsibility
and a system prompt that enforces the hierarchical contract:

> *"You are a subordinate worker. You take instructions from the manager and
> report back to the manager. Do not communicate with other workers."*


In [11]:
from langchain.agents import create_agent

# ── Worker sub-agents ──────────────────────────────────────────────────────
# Workers have no tools — they reason over the catalog passed in the message.
# The hierarchical contract is enforced in the system prompt.

destination_worker = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[],
    system_prompt=(
        "You are the Destination Worker in a hierarchical travel-agency MAS.\n"
        "You receive instructions from the manager and report back to the manager only.\n"
        "Responsibility: choose the best destination and travel period.\n"
        "Only use destinations from the catalog. Be concise and decisive."
    ),
)

booking_worker = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[],
    system_prompt=(
        "You are the Booking Worker in a hierarchical travel-agency MAS.\n"
        "You receive instructions from the manager and report back to the manager only.\n"
        "Responsibility: choose one flight and one hotel that fit the destination and budget.\n"
        "Only use flights and hotels from the catalog. Be concise and decisive."
    ),
)

activity_worker = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[],
    system_prompt=(
        "You are the Activity Worker in a hierarchical travel-agency MAS.\n"
        "You receive instructions from the manager and report back to the manager only.\n"
        "Responsibility: choose activities that match the traveler's food and culture interests.\n"
        "Only use activities from the catalog. Be concise and decisive."
    ),
)

print("Worker sub-agents created.")


Worker sub-agents created.


## 4 · Worker Tools (Delegation via State)

Each tool implements the **manager → worker → manager** communication channel:

1. **Read context** — pull `manager_notes` from state so the worker sees earlier results.
2. **Delegate** — invoke the worker sub-agent with a message that includes the manager's instruction and context.
3. **Log result** — return `Command(update={"manager_notes": updated_notes, ...})`.

> **Key hierarchical property:** workers read context from `manager_notes`
> (the manager's log), never from each other directly.


In [12]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command


@tool
async def delegate_to_destination_worker(runtime: ToolRuntime) -> str:
    '''Manager delegates destination selection to the Destination Worker.'''
    notes = runtime.state.get("manager_notes", [])
    notes_text = "\n".join(notes) if notes else "No prior notes."

    # The worker receives: request + catalog + manager's notes so far
    response = await destination_worker.ainvoke({
        "messages": [HumanMessage(content=(
            f"Traveler request:\n{USER_REQUEST}\n\n"
            f"Catalog:\n{CATALOG_TEXT}\n\n"
            f"Manager notes so far:\n{notes_text}\n\n"
            "Your task: choose the best destination and travel period. Report back to the manager."
        ))]
    })
    result = response["messages"][-1].content
    new_entry = f"destination_worker:\n{result}"
    return Command(update={
        "manager_notes": [new_entry],
        "messages": [ToolMessage(result, tool_call_id=runtime.tool_call_id)],
    })


@tool
async def delegate_to_booking_worker(runtime: ToolRuntime) -> str:
    '''Manager delegates flight and hotel selection to the Booking Worker.'''
    notes = runtime.state.get("manager_notes", [])
    notes_text = "\n".join(notes) if notes else "No prior notes."

    response = await booking_worker.ainvoke({
        "messages": [HumanMessage(content=(
            f"Traveler request:\n{USER_REQUEST}\n\n"
            f"Catalog:\n{CATALOG_TEXT}\n\n"
            f"Manager notes so far:\n{notes_text}\n\n"
            "Your task: choose one flight and one hotel. Report back to the manager."
        ))]
    })
    result = response["messages"][-1].content
    new_entry = f"booking_worker:\n{result}"
    return Command(update={
        "manager_notes": [new_entry],
        "messages": [ToolMessage(result, tool_call_id=runtime.tool_call_id)],
    })


@tool
async def delegate_to_activity_worker(runtime: ToolRuntime) -> str:
    '''Manager delegates activity selection to the Activity Worker.'''
    notes = runtime.state.get("manager_notes", [])
    notes_text = "\n".join(notes) if notes else "No prior notes."

    response = await activity_worker.ainvoke({
        "messages": [HumanMessage(content=(
            f"Traveler request:\n{USER_REQUEST}\n\n"
            f"Catalog:\n{CATALOG_TEXT}\n\n"
            f"Manager notes so far:\n{notes_text}\n\n"
            "Your task: choose 2-3 activities (food + culture mix). Report back to the manager."
        ))]
    })
    result = response["messages"][-1].content
    new_entry = f"activity_worker:\n{result}"
    return Command(update={
        "manager_notes": [new_entry],
        "messages": [ToolMessage(result, tool_call_id=runtime.tool_call_id)],
    })

print("Worker delegation tools defined.")


Worker delegation tools defined.


## 5 · Manager Agent

The manager is the authoritative coordinator. Its system prompt encodes the
**plan → delegate → synthesise** pattern:

1. Delegate destination first (so later workers know where they're working).
2. Delegate booking (flight + hotel) using the destination result.
3. Delegate activities last.
4. Synthesise all worker outputs into the final client itinerary.


In [13]:
from langchain.agents import create_agent

manager = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[
        delegate_to_destination_worker,
        delegate_to_booking_worker,
        delegate_to_activity_worker,
    ],
    state_schema=HierarchicalState,
    system_prompt=(
        "You are the Manager in a hierarchical travel-agency MAS.\n"
        "You control the workflow from start to finish.\n\n"
        "Follow this exact order:\n"
        "1. Call delegate_to_destination_worker — destination must be chosen first.\n"
        "2. Call delegate_to_booking_worker — uses the destination chosen in step 1.\n"
        "3. Call delegate_to_activity_worker — uses the destination chosen in step 1.\n"
        "4. Once all three workers have reported, synthesise their outputs into one "
        "   clean, client-ready travel itinerary. Do not call any more tools after that."
    ),
)

print("Hierarchical manager created.")


Hierarchical manager created.


## 6 · Run

In [14]:
from langchain.messages import HumanMessage

response = await manager.ainvoke(
    {
        "messages": [HumanMessage(content=USER_REQUEST)],
        "manager_notes": [],  # starts empty; grows as workers report back
    }
)


In [15]:
from pprint import pprint
pprint(response)


{'manager_notes': ['destination_worker:\n'
                   'Recommend a 4-day spring trip to Prague from Rome in '
                   'April–June. It offers direct, affordable flights (FR-721 '
                   'at EUR 110, 1h 55m), a historic and compact city ideal for '
                   'easy sightseeing, and mid-range hotel options like Old '
                   'Town House at EUR 115/night in a central location. The '
                   'trip combines culture (Old Town walking tour, EUR 18) and '
                   'food experiences (Czech dinner and jazz night, EUR 30) '
                   'with a simple, relaxed itinerary suitable within a '
                   'mid-range budget.',
                   'booking_worker:\n'
                   'Choose the direct flight FR-721 at EUR 110 and the Old '
                   'Town House hotel at EUR 115/night for a centrally located, '
                   'mid-range hotel option. This fits the historic, compact '
                   'vib

In [16]:
print(response["messages"][-1].content)


Here is a clean, client-ready 4-day spring travel itinerary from Rome to Prague, designed for a mid-range budget with easy flights, a central hotel, and a balanced mix of food and culture:

Day 1: Arrival & Settle In
- Flight: Direct flight FR-721 from Rome to Prague (EUR 110, approximately 1h 55m)
- Accommodation: Check into Old Town House hotel, centrally located (EUR 115/night)
- Evening: Relax and enjoy a casual walk around the Old Town Square to absorb the local vibe

Day 2: Cultural Exploration
- Morning: Guided Old Town walking tour to explore Prague’s historic sites (EUR 18)
- Afternoon: Free time for leisure or optional café visit in the Old Town area
- Evening: Enjoy a Czech dinner coupled with a jazz night experience (EUR 30)

Day 3: Leisure & Local Delights
- Morning and afternoon: Free day for self-guided exploration, shopping, or visiting additional attractions such as Prague Castle or Charles Bridge
- Evening: Sample local street food or dine at a recommended mid-range r

## 7 · Key Takeaways

| What you saw | Why it matters |
|---|---|
| `HierarchicalState` with `manager_notes` | The manager's log — workers report here, never to each other |
| Workers read context from `notes_text` | Context flows manager → worker, not worker → worker |
| Tools called in order by the manager | The manager controls the workflow sequence |
| Manager synthesises the final answer | One agent is responsible for quality and coherence |
